In [1]:
import pandas as pd
import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel
import spacy
import nltk
from nltk.corpus import stopwords
import re
import matplotlib.pyplot as plt
from pathlib import Path 


In [2]:
# Download NLTK stopwords
nltk.download('stopwords')
stop_words = stopwords.words('english')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\thenu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [4]:
DIRECTORY = Path.cwd().parent / "csv"
data = pd.read_csv(DIRECTORY / "macro_community_groups.csv")

# Preprocess the text data
def preprocess_text(text):
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # Remove non-alphabet characters
    text = re.sub(r'\S*@\S*\s?', '', text)    # Remove emails
    text = re.sub(';', '', text)  # Remove ;
    text = re.sub('\'', '', text)  # Remove apostrophes
    text = re.sub(r'\s+', ' ', text)          # Remove extra spaces
    text = text.lower()  # Convert to lowercase
    return text

In [5]:
data['PaperTitles'] = data['PaperTitles'].apply(preprocess_text)
data['Abstracts'] = data['Abstracts'].apply(preprocess_text)
data['CombinedText'] = data['PaperTitles'] + ' ' + data['Abstracts']
data.drop(columns=['PaperTitles', 'Abstracts', 'DOIs'], inplace=True)
data.head()

,Macro_Community,CombinedText
0,0,an overview of microsoft academic service mas ...
1,1,graph embedding for mapping interdisciplinary ...


In [6]:
# data.to_csv(DIRECTORY / "preprocessed_macro_community_groups.csv", index=False)

In [7]:
# Tokenize and remove stopwords
def tokenize(text):
    tokens = gensim.utils.simple_preprocess(text, deacc=True)
    tokens = [token for token in tokens if token not in stop_words]
    return tokens

data['tokens'] = data['CombinedText'].apply(tokenize)

In [8]:
# Lemmatization using spaCy
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
def lemmatize(tokens):
    doc = nlp(" ".join(tokens))
    return [token.lemma_ for token in doc]

data['lemmas'] = data['tokens'].apply(lemmatize)

In [22]:
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# -----------------------------
# 1) Aggregate words per macro community
# -----------------------------
community_texts = (
    data.groupby("Macro_Community")["lemmas"]
    .apply(lambda x: [w for doc in x for w in doc])
)

texts = community_texts.tolist()

# -----------------------------
# 2) Dictionary + filtering
# -----------------------------
id2word = corpora.Dictionary(texts)

id2word.filter_extremes(
    no_below=1,
    no_above=0.9
)

assert len(id2word) > 0, "Dictionary is empty — check filtering thresholds"

corpus = [id2word.doc2bow(text) for text in texts]

# -----------------------------
# 3) Train LDA (global topics)
# -----------------------------
NUM_TOPICS = 5
lda_model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=50,
    iterations=500,
    alpha="asymmetric",
    eta="symmetric"
)

# -----------------------------
# 4) Display: per community -> top topics + keywords
# -----------------------------
TOP_K_TOPICS_PER_COMMUNITY = 2   # change this as you like
TOP_N_KEYWORDS = 5               # change this as you like

for macro_id, bow in zip(community_texts.index, corpus):
    # get topic distribution for this community
    topics = lda_model.get_document_topics(bow, minimum_probability=0.0)
    topics_sorted = sorted(topics, key=lambda x: x[1], reverse=True)

    print(f"\nMacro Community {macro_id}")
    for topic_id, prob in topics_sorted[:TOP_K_TOPICS_PER_COMMUNITY]:
        keywords = [w for w, _ in lda_model.show_topic(topic_id, topn=TOP_N_KEYWORDS)]
        print(f"  Topic {topic_id} (p={prob:.4f}): {', '.join(keywords)}")

# -----------------------------
# 5) Coherence (global)
# -----------------------------
coherence_model = CoherenceModel(
    model=lda_model,
    texts=texts,
    dictionary=id2word,
    coherence="c_v"
)

print("\nGlobal Coherence:", coherence_model.get_coherence())



Macro Community 0
  Topic 2 (p=0.9995): kernel, manifold, biogrid, cqa, deepwalk
  Topic 0 (p=0.0002): biomedical, bert, corpus, gnn, nlp

Macro Community 1
  Topic 4 (p=0.9996): biomedical, bert, nlp, corpus, gnn
  Topic 0 (p=0.0001): biomedical, bert, corpus, gnn, nlp

Global Coherence: 0.5171369214008041
